In [ ]:
words = open('names.txt', 'r').read().splitlines()


In [ ]:
chars = sorted(list(set(''.join(words))))
stoi = {s : i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i : s for s, i in stoi.items()}

In [ ]:
import torch

In [ ]:
#created traing set of biagrams (x, y)
xs = []
ys = []

for w in words[:3]:
    chs = ['.'] + list(w) + ['.']
    for ch1 , ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)




In [ ]:
xs

In [ ]:
ys

In [ ]:
import torch.nn.functional as F
xenc = F.one_hot(xs, num_classes=27).float()
xenc.shape

In [ ]:
import matplotlib.pyplot  as plt
plt.imshow(xenc)

In [ ]:
#randomnly initialize 27 neurons's weights, each neuron receive 27 inputs 
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g)

In [ ]:
xenc = F.one_hot(xs, num_classes=27).float() #input to each network, one hot encoding
logits = xenc @ W #tensor.dot works only in 1D vector unlike np.dot, predicts log counts
counts = logits.exp() #count equivalent to N[] , initially fake count, untill optimizes
probs = counts / counts.sum(1, keepdim=True)
#btw: last 2 lines here are together called a softmax

In [ ]:
probs.shape

In [ ]:
nlls = torch.zeros(5) #negative log loss
for i in range(5):
    #i - th bigram
    x = xs[i].item() # input character index
    y = ys[i].item() #target/label character index
    print('-----------')
    print(f'bigram example {i + 1}: {itos[x]}-{itos[y]} (indexes {x},{y})')
    print('input to the neural net:', x)
    print('output probabilities from the neural net:', probs[i])
    print(f'label (actual next character): {y}')
    p = probs[i, y]
    print('probability assigned by the net to the correct character:', p.item())
    logp = torch.log(p)
    print('log likelihood:', logp.item())
    nll = -logp
    print('negative log likelihood', nll.item())
    nlls[i] = nll

print('================')
print('average negative likelihood, i.e. loss = ', nlls.mean().item())




In [ ]:
# --------------- !!! optimization !!! ------------
#randomnly initialize 27 neurons's weights, each neuron receive 27 inputs 
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

In [ ]:
#forward pass
xenc = F.one_hot(xs, num_classes=27).float() #input to each network, one hot encoding
logits = xenc @ W #tensor.dot works only in 1D vector unlike np.dot, predicts log counts
counts = logits.exp() #count equivalent to N[] , initially fake count, untill optimizes
probs = counts / counts.sum(1, keepdim=True)
#btw: last 2 lines here are together called a softmax

In [ ]:
xs

In [ ]:
ys

In [ ]:
probs.shape
probs[0, 5] , probs[1, 13], probs[2, 13], probs[3, 1]

In [ ]:
probs[torch.arange(4), ys[:4]]

In [ ]:
loss = -probs[torch.arange(5), ys[:5]].log().mean()
loss

In [ ]:
#backward pass
W.grad = None #set the zero gradient
loss.backward()

In [ ]:
W.grad.dtype
#each element of W.grad tells influence on the loss function
# +ve value would increase loss and vice versa, hence we w -= alpha * dw

In [ ]:
W.data += -0.1 * W.grad

In [ ]:
# ---- !!! optimization !!! completely, this time actually


In [ ]:
#create dataset
xs, ys = [], []

for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1 , ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)


In [ ]:
#intialize network
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)


In [ ]:
num = xs.nelement()

In [95]:
#gradient descent
for k in range(100):
    #forward pass
    xenc = F.one_hot(xs, num_classes=27).float()
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)
    loss = -probs[torch.arange(num), ys].log().mean() + 0.01 * (W ** 2).mean() #l2 regularization
    #print(loss.item())

    #backward pass 
    W.grad = None
    loss.backward()

    #update
    W.data += -60.0 * W.grad

print(loss.item())
 

2.482208251953125


In [99]:
g = torch.Generator().manual_seed(2147483647)
for i in range(5):
    out = []
    ix = 0
    while True:
        xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
        logits = xenc @ W
        counts = logits.exp()
        probs = counts / counts.sum(1, keepdim=True)
        ix = torch.multinomial(probs, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if (ix == 0):
            break
    print(''.join(out))

cexze.
momasurailezityha.
konimittain.
llayn.
ka.
